## 数据预测

### 人口预测

In [ ]:
import pandas as pd
import numpy as np


def load_data(age_file, fertility_file, mortality_file):
    """加载人口分布、生育率和死亡率数据"""
    age_distribution = pd.read_csv(age_file)
    fertility_rate_forecast = pd.read_csv(fertility_file)
    mortality_rate = pd.read_csv(mortality_file)
    return age_distribution, fertility_rate_forecast, mortality_rate


def calculate_newborns(fertility_rate, age_distribution, female_ratio=0.5, generation_time=30):
    """根据总和生育率计算新生儿数量"""
    fertile_population = age_distribution.loc[(age_distribution['age'] >= 15) &
                                              (age_distribution['age'] <= 49), '2020']
    female_population = fertile_population.sum() * female_ratio
    newborns = female_population * fertility_rate / generation_time
    return newborns


def apply_mortality(age_distribution, mortality_rate):
    """根据死亡率更新年龄段人口"""
    age_distribution['mortality_rate'] = age_distribution['age'].map(
        dict(zip(mortality_rate['age'], mortality_rate['mortality_rate']))
    )
    age_distribution['2020'] = age_distribution['2020'] * \
        (1 - age_distribution['mortality_rate'])
    return age_distribution


def simulate_population_forward(age_distribution, fertility_rate_forecast, mortality_rate, years):
    """模拟未来每年的各年龄段人口"""
    results = pd.DataFrame(index=age_distribution['age'], columns=years)

    for year in years:
        fertility_rate = fertility_rate_forecast.loc[fertility_rate_forecast['year']
                                                     == year, 'fertility_rate'].values[0]

        # 计算新生儿数量
        newborns = calculate_newborns(fertility_rate, age_distribution)

        # 年龄推进并考虑新生儿
        next_population = age_distribution['2020'].shift(
            1, fill_value=newborns)
        age_distribution['2020'] = next_population

        # 应用死亡率
        age_distribution = apply_mortality(age_distribution, mortality_rate)

        # 保存结果
        results[year] = age_distribution['2020'].astype(int)

    return results


def simulate_population_backward(age_distribution, years):
    """反向推算每年的各年龄段人口（假设每年人口只往前推移一岁）"""
    results = pd.DataFrame(index=age_distribution['age'], columns=years)

    for year in years:
        # 将每个年龄段的群体向前推移一岁
        age_distribution['2020'] = age_distribution['2020'].shift(-1, fill_value=0)

        # 保存结果
        results[year] = age_distribution['2020'].astype(int)

    return results


def main():
    # 文件路径
    age_data_file = '../data/population_age_distribution.csv'
    fertility_rate_file = '../data/fertility_rate_forecast.csv'
    mortality_rate_file = '../data/mortality_rate.csv'

    # 加载数据
    age_distribution, fertility_rate_forecast, mortality_rate = load_data(
        age_data_file, fertility_rate_file, mortality_rate_file
    )

    # 获取预测年份
    forecast_years = fertility_rate_forecast['year'].values

    # 模拟未来人口数据（从2021年到2200年）
    future_population = simulate_population_forward(
        age_distribution.copy(), fertility_rate_forecast, mortality_rate, forecast_years)

    # 反向推算人口数据（从2019年到1950年）
    past_years = np.arange(2019, 1950, -1)  # 反推的年份范围
    past_population = simulate_population_backward(
        age_distribution.copy(), past_years)

    # 合并未来和过去的人口数据
    all_population = pd.concat([past_population, age_distribution.iloc[:, 1], future_population.iloc[:, 1:]], axis=1)

    # 保存结果
    all_population.to_csv(
        '../data/full_population_projection.csv', index_label='age')
    print("过去和未来每年的预期各年龄人口数据已生成：full_population_projection.csv")


if __name__ == "__main__":
    main()

## 数据可视化

### 总人口

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 计算每年总人口
df = pd.read_csv('../data/full_population_projection.csv')
total_population = df.drop('age', axis=1).sum()

# 绘制总人口曲线图
plt.figure(figsize=(10, 6))
plt.plot(total_population.index, total_population.values, marker='o', color='r', label='Total Population')
plt.xlabel('Year')
plt.ylabel('Total Population')
plt.title('Total Population Over Years')
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.show()

### 中位数

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV数据
df = pd.read_csv('../data/full_population_projection.csv')

# 计算每年年龄的中位数
median_ages = []
for year in df.columns[1:]:  # 跳过'age'列
    populations = df[year]
    cumulative_population = populations.cumsum()  # 累积人口
    total_population = cumulative_population.iloc[-1]  # 总人口
    median_threshold = total_population / 2
    # 找到达到总人口一半的年龄
    median_age = np.searchsorted(cumulative_population, median_threshold)
    median_ages.append(df['age'][median_age])

# 绘制各年年龄中位数曲线
plt.figure(figsize=(10, 6))
plt.plot(df.columns[1:], median_ages, marker='o', color='b', label='Median Age')
plt.xlabel('Year')
plt.ylabel('Median Age')
plt.title('Median Age Over Years')
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.show()

### 抚养比

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 读取CSV数据
df = pd.read_csv('../data/full_population_projection.csv')

# 将第一行作为列名，并将第一列作为索引
df = pd.read_csv('../data/full_population_projection.csv', header=0, index_col=0)

# 按列名（年份）排序
df = df.sort_index(axis=1)

# 计算非劳动年龄人口和劳动年龄人口
children = df.loc[(df.index >= 0) & (df.index <= 14)].sum()  # 0-14岁
adults = df.loc[(df.index >= 15) & (df.index <= 64)].sum()   # 15-64岁
elderly = df.loc[df.index >= 65].sum()                       # 65岁及以上

# 计算依赖比率
dependency_ratio = (children + elderly) / adults * 100

# 计算幼儿抚养比
child_dependency_ratio = children / adults * 100

# 将年份索引转换为整数类型
dependency_ratio.index = dependency_ratio.index.astype(int)
child_dependency_ratio.index = child_dependency_ratio.index.astype(int)

# 筛选 2020 年及以后且小于 2100 年的数据
dependency_ratio_filtered = dependency_ratio[(dependency_ratio.index >= 2020) & (dependency_ratio.index < 2100)]
child_dependency_ratio_filtered = child_dependency_ratio[(child_dependency_ratio.index >= 2020) & (child_dependency_ratio.index < 2100)]

# 绘制历年抚养比曲线图
plt.figure(figsize=(10, 5))  # 增加图表宽度
plt.plot(dependency_ratio_filtered.index, dependency_ratio_filtered.values, marker='o', color='g', label='Dependency Ratio')
plt.plot(child_dependency_ratio_filtered.index, child_dependency_ratio_filtered.values, marker='o', color='b', label='Child Dependency Ratio')
plt.xlabel('Year')
plt.ylabel('Ratio (%)')
plt.title('Dependency Ratio and Child Dependency Ratio Over Years (2020-2099)')
plt.grid(True)
plt.legend()

# 优化 x 轴标签显示：每隔 5 年显示一次年份
years = dependency_ratio_filtered.index  # 获取所有年份
plt.xticks(years[::5], rotation=45, ha='right')  # 每隔 5 年显示一次，旋转 45 度并右对齐
plt.tight_layout()  # 自动调整布局，避免标签被截断
plt.show()

### 漏斗图视频

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 选择合适的字体，如 SimHei
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

# 读取CSV数据
df = pd.read_csv('../data/full_population_projection.csv')

# 创建保存图像的目录
output_dir = 'population_distribution_images'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 要标注的年龄和对应的标签
annotations = {
    26: "公寓",
    31: "婚房",
    42: "改善房",
    46: "装修",
    51: "大学费",
    53: "汽车",
    60: "医疗",
    65: "养老",
    70: "旅行",
    77: "药物",
    84: "养老院"
}

# 为每一年绘制人口分布图并保存
for year in df.columns[1:]:  # 跳过'age'列
    plt.figure(figsize=(10, 6))

    # 获取年龄和对应的人口数据
    ages = df['age']
    populations = df[year]

    # 计算总人口
    total_population = populations.sum()

    # 计算人口的相对波动（可以选择与最大值或前一年比较）
    max_population = populations.max()
    normalized_populations = populations / max_population  # 归一化到最大值范围

    # 使用颜色映射基于人口相对值来着色
    norm = mcolors.Normalize(vmin=normalized_populations.min(), vmax=normalized_populations.max())
    cmap = plt.get_cmap('coolwarm')  # 你可以选择其他的颜色映射，例如 'viridis', 'plasma', 'inferno'

    # 绘制年龄分布图并应用相应颜色
    bars = plt.bar(ages, populations, color=cmap(norm(normalized_populations)))

    # 添加颜色条显示相对值
    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array(normalized_populations)
    cbar = plt.colorbar(mappable)
    cbar.set_label('Relative Population')

    plt.xlabel('Age')
    plt.ylabel('Population')
    plt.title(f'Population Distribution in {year}')
    plt.grid(True)

    # 显示总人口和中位年龄
    cumulative_population = populations.cumsum()  # 累积人口
    median_threshold = total_population / 2
    median_age = np.searchsorted(cumulative_population, median_threshold)
    median_age_value = ages[median_age]  # 获取中位年龄对应的年龄值
    # 将年轻人和老人分两段，计算养老抚养比
    young_population = populations[ages < 60].sum()
    old_population = populations[ages >= 60].sum()
    dependency_ratio = old_population / young_population

    # 在图表上显示总人口、中位年龄、养老抚养比
    plt.text(0.8, 0.9, f'Total Population: {total_population:,}',
             horizontalalignment='center', verticalalignment='center',
             transform=plt.gca().transAxes, fontsize=12, color='black', fontweight='bold')
    plt.text(0.8, 0.85, f'Median Age: {median_age_value}',
             horizontalalignment='center', verticalalignment='center',
             transform=plt.gca().transAxes, fontsize=12, color='black', fontweight='bold')
    plt.text(0.8, 0.8, f'Dependency Ratio: {dependency_ratio:.2f}',
             horizontalalignment='center', verticalalignment='center',
             transform=plt.gca().transAxes, fontsize=12, color='black', fontweight='bold')

    # 在指定的年龄位置添加标注
    for age, label in annotations.items():
        if age in ages.values:
            plt.text(age, populations[ages == age], label,
                     horizontalalignment='center', verticalalignment='bottom',
                     fontsize=10, color='black', fontweight='bold')

    # 保存每年的人口分布图为PNG文件
    plt.savefig(f'{output_dir}/population_distribution_{year}.png')
    plt.close()

print("所有人口分布图已生成并保存！")

In [ ]:
import cv2
import os

# 定义图片所在的目录和视频输出文件名
image_dir = 'population_distribution_images'
output_video = '../data/population_distribution_video.mp4'

# 获取所有图片文件（按照文件名排序）
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

# 读取第一张图片来获取尺寸（假设所有图片尺寸相同）
first_image_path = os.path.join(image_dir, image_files[0])
first_image = cv2.imread(first_image_path)
height, width, layers = first_image.shape

# 设置视频写入器
fourcc = cv2.VideoWriter_fourcc(*'X264')  # 使用 H.264 编码器（X264）

frame_rate = 3  # 设置帧率为25帧每秒，较高的帧率使视频更流畅
bitrate = 5000  # 设定比特率为 5000 kbps

# 设置视频输出文件，调整帧率和分辨率
video_writer = cv2.VideoWriter(output_video, fourcc, frame_rate, (width, height))

# 将所有图片按顺序添加到视频中
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    image = cv2.imread(image_path)
    video_writer.write(image)  # 写入图片到视频

# 释放视频写入器
video_writer.release()

print(f"视频已生成：{output_video}")